# 1. Walidacja i podziały

Celem jest oszacowanie prawdopodobieństwa zdarzenia `default`. Horyzont i dostępność cech w momencie decyzji wymagają potwierdzenia w dokumentacji danych; nie zakładamy automatycznie 12 miesięcy.

Dane przekrojowe dzielimy stratyfikowanie na train/validation/test. Transformacje uczą się tylko na train (a podczas CV — na foldzie treningowym). Test pozostaje do końcowej oceny modeli.

In [ ]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from scipy.special import logit, expit
from scipy.stats import binomtest, beta as beta_dist, spearmanr
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, log_loss, brier_score_loss

import os

print("Python:", sys.version.split()[0], "| numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__, "| matplotlib:", matplotlib.__version__)
warnings.filterwarnings("ignore", category=FutureWarning)
RNG_SEED = 42
TARGET_PD = 0.04

In [ ]:
from pathlib import Path
import unicodedata

# Uruchomienie z katalogu projektu albo z jego podkatalogu.
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "project_desc.md").exists() and (p / "data").is_dir())
DATA_DIR = PROJECT_DIR / "data"
# Normalizacja nazwy obsługuje oba sposoby zapisu polskich znaków na macOS.
csv_path = next(p for p in DATA_DIR.glob("*.csv")
                if unicodedata.normalize("NFC", p.name) == "zbiór_2.csv")

In [ ]:
df = pd.read_csv(csv_path)
assert df["default"].notna().all() and set(df["default"].unique()) == {0, 1}
df.head(10)

Zbiór zawiera zmienne opisujące sytuację finansową przedsiębiorstw, m.in. pozycje bilansowe, dane z rachunku zysków i strat, przepływy pieniężne oraz wskaźniki finansowe. Występują w nim również zmienne kategorialne opisujące formę własności, formę prawną i rodzaj działalności.

In [ ]:
duplikaty = df.drop(columns=['default']).duplicated().sum()
print("Liczba zduplikowanych wierszy:", duplikaty)

subset_cols = ['wsk_liczba_dni_istnienia', 'pkdKod', 'formaWlasnosci_Symbol', 'szczegolnaFormaPrawna_Symbol']
duplikaty = df.duplicated(subset=subset_cols, keep=False)

print("Liczba wierszy posiadających duplikujący się zestaw (PKD, l. dni istnienia, symbol formy włsności i symbol szczególnej formy prawnej):", duplikaty.sum())

Brak identycznych wierszy nie dowodzi, że każda obserwacja dotyczy innej firmy. Kombinacja PKD, wieku i formy prawnej również nie jest identyfikatorem podmiotu. Przy braku identyfikatora stosujemy podział stratyfikowany, dokumentując to ograniczenie; jeśli otrzymamy identyfikator firmy, należy sprawdzić konieczność podziału grupowego.

# Podział i walidacja danych
1. **Podział główny (Train / Validation / Test):**
   - Zbiór Treningowy (70%) -> modelowanie i dopasowanie transformacji.
   - Zbiór Walidacyjny (15%) -> strojenie hiperparametrów.
   - Zbiór Testowy (15%) -> ostateczna ocena modelu.
2. **Cross-Validation (Stratified 5-Fold):** Stosowany na zbiorze treningowym do stabilnej oceny modeli bazowych.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

X = df.drop(columns=["default"])
y = df["default"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RNG_SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85,
    stratify=y_temp,
    random_state=RNG_SEED
)

#do późniejszego podziału przy modelach
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RNG_SEED
)

# EDA wyłącznie na train; validation i test pozostają surowe.
df_temp = pd.concat([X_train, y_train], axis=1).copy()
df_test = pd.concat([X_test, y_test], axis=1).copy()

print("Dane → train (70%) → Stratified 5-fold CV")
print("     → validation (15%): wybór modelu / kalibracja bez testu")
print("     → test (15%): wyłącznie końcowa ocena")

# Sprawdzenie poprawności podziału

In [ ]:
print("Train — liczebności klas:")
print(y_train.value_counts())
print("Train — udział klas (%):")
print(y_train.value_counts(normalize=True) * 100)

In [ ]:
for name, target in [("Train", y_train), ("Validation", y_val)]:
    print(f"\n{name}")
    print("Liczba obserwacji:", len(target))
    print("Default = 1:", target.sum())
    print("Udział default = 1:", round(target.mean() * 100, 2), "%")
print("Liczba obserwacji testowych:", len(X_test))

Zastosowana stratyfikacja zachowała strukturę zmiennej objaśnianej default.

# Sprawdzenie przecieku

In [ ]:
cechy_wszystkie = [c for c in df.columns if c not in {"default"}]
rows = []
for c in cechy_wszystkie:
    if not pd.api.types.is_numeric_dtype(df[c]):
        continue
    m = df_temp[c].notna() & np.isfinite(df_temp[c])
    if df_temp.loc[m, "default"].nunique() < 2:
        continue
    auc = roc_auc_score(df_temp.loc[m, "default"], df_temp.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

Jednowymiarowe AUC jest diagnostyką, a nie dowodem braku leakage. Dostępność i sposób wyliczenia cech należy zweryfikować w opisie zmiennych. Szczególnie silne zależności wymagają sprawdzenia, ale niski AUC nie wyklucza przecieku.

# **2. Przygotowanie i EDA**

Celem niniejszego etapu jest ocena jakości danych, poznanie rozkładów zmiennych oraz identyfikacja zależności pomiędzy zmiennymi objaśniającymi a zmienną default. Na podstawie przeprowadzonych analiz zostaną określone metody przygotowania cech do dalszego modelowania.

# Struktura i typy zmiennych

In [ ]:
print("Rozmiar:", df_temp.shape)

print("\nInformacje o danych:")
df_temp.info()

summary = pd.DataFrame({
    "dtype": df_temp.dtypes.astype(str),
    "n_unique": df_temp.nunique(),
    "missing_count": df_temp.isna().sum(),
    "missing_pct": df_temp.isna().mean() * 100
})

summary = summary.sort_values(
    "missing_pct", ascending=False
)

display(summary)

constant_cols = [
    col for col in df_temp.columns
    if df_temp[col].nunique(dropna=False) <= 1
]

print("Zmienne stałe:", constant_cols)

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns

inf_report = pd.DataFrame({
    "inf_plus": np.isposinf(X_train[num_cols]).sum(),
    "inf_minus": np.isneginf(X_train[num_cols]).sum()
})

display(inf_report[
    (inf_report["inf_plus"] > 0) |
    (inf_report["inf_minus"] > 0)
])

Mamy wartości nieskończone, co trzeba będzie uwzględnić w dalszych etapach.

# Korelacje z default

In [ ]:
kor_def = df_temp.select_dtypes(include=[np.number]).corr()['default'].sort_values(ascending=False)
print("Cechy najsilniej skorelowane z defaultem(na plusie):")
print(kor_def.head(7))
print("\nCechy najsilniej skorelowane z defaultem(na minusie):")
print(kor_def.tail(7))

Zależności z defaultem opisujemy na zbiorze treningowym. Znaczenie indeksów `wsk_index_nowe` i `wsk_index_skr_zaw` należy odczytać z opisu zmiennych — sama nazwa nie uzasadnia interpretacji jako wieku firmy. Korelacje cech mających bardzo dużo braków opierają się na małej próbie i wymagają ostrożności. Zależności nie stanowią dowodu przyczynowości.

In [ ]:
missing_by_default = (
    X_train.assign(default=y_train)
    .groupby("default")
    [X_train.columns].apply(lambda x: x.isna().mean() * 100)
    .T
)

missing_by_default.columns = [
    "Braki_default_0",
    "Braki_default_1"
]

display(missing_by_default.sort_values(
    "Braki_default_1",
    ascending=False
))

# Rozkłady wybranych cech

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

wybrane_cechy = [
    'wsk_index_nowe',
    'wsk_index_skr_zaw',
    'wsk_pokrycie_wyd_fin_gotowkowe_2',
    'przychody_sprzedazy'
]

plt.figure(figsize=(14, 10))
for i, col in enumerate(wybrane_cechy, 1):
  plt.subplot(2, 2, i)
  sns.histplot(df_temp[col].dropna(), kde=True, bins=30, color='royalblue')
  plt.title(f'Rozkład zmiennej: {col}', fontsize=11)
  plt.xlabel('')
  plt.ylabel('Liczba firm')

plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 4))
for i, col in enumerate(['Aktywa', 'wsk_zadluzenia'], 1):
  plt.subplot(1, 2, i)
  sns.boxplot(x=df_temp[col], color='orange')
  plt.title(f'Boxplot (wartości odstające): {col}', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
for col in wybrane_cechy:
    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=pd.DataFrame({
            col: X_train[col],
            "default": y_train
        }),
        x="default",
        y=col
    )

    plt.title(f"{col} – porównanie według default")
    plt.xlabel("Default")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd

num_features = X_train.select_dtypes(include=[np.number]).columns
corr_matrix = X_train[num_features].corr().abs()

upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = []
for col in upper_tri.columns:
  high_corr = upper_tri.index[upper_tri[col] > 0.85].tolist()
  for row in high_corr:
    high_corr_pairs.append(
        {'Cecha 1': row, 'Cecha 2': col, 'Korelacja': corr_matrix.loc[row, col]}
    )

df_high_corr = pd.DataFrame(high_corr_pairs).sort_values(
    'Korelacja', ascending=False
)

print(f'Liczba par cech silnie skorelowanych (>0.85): {len(df_high_corr)}')
display(df_high_corr)

Usunięcie stałych i identycznych kolumn będzie częścią pipeline’u poniżej. Lista jest wyznaczana na train, a podczas CV od nowa na każdym foldzie. Surowe `X_train`, `X_val`, `X_test` pozostają bez zmian.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class UsunStaleIdentyczne(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        self.columns_ = []
        self.removed_ = []
        for col in X.columns:
            if X[col].nunique(dropna=False) <= 1 or any(
                X[col].equals(X[kept]) for kept in self.columns_
            ):
                self.removed_.append(col)
            else:
                self.columns_.append(col)
        return self

    def transform(self, X):
        return X.loc[:, self.columns_].copy()

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columns_, dtype=object)

kontrola_kolumn = UsunStaleIdentyczne().fit(X_train)
print("Kolumny do usunięcia:", kontrola_kolumn.removed_)
print("Liczba pozostałych cech:", len(kontrola_kolumn.columns_))

# Brakujące wartości

In [ ]:
import seaborn as sns

missing_report = pd.DataFrame({
    "missing_count": df_temp.isna().sum(),
    "missing_pct": df_temp.isna().mean() * 100
})

missing_report = missing_report.sort_values(
    "missing_pct",
    ascending=False
)
display(missing_report)

missing_plot = missing_report[
    missing_report["missing_count"] > 0
].head(40)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=missing_plot.reset_index(),
    x="missing_pct",
    y="index"
)

plt.xlabel("Braki danych proc.")
plt.ylabel("Zmienna")
plt.title("Zmienne z największymi brakami danych")
plt.tight_layout()
plt.show()

Część zmiennych ma ponad 90% braków danych. Zachowujemy imputację medianą i flagi braków dla cech liczbowych oraz najczęstszą wartość dla kategorii — według wzorca z Lab 3, komórka 19. Imputery są dopasowywane dopiero wewnątrz pipeline’u. Nieskończoności traktujemy jako braki.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Źródło: labs/Lab_3_rating_projekt1.ipynb, komórka 19 (indeks od 0).
# Oryginalny układ imp → sc oraz imp → ohe; poniżej dostosowanie do naszych cech.
def ohe(**kw):
    try:
        return OneHotEncoder(sparse_output=False, **kw)
    except TypeError:
        return OneHotEncoder(sparse=False, **kw)

CAT_CODES = ["szczegolnaFormaPrawna_Symbol", "formaWlasnosci_Symbol", "pkdKod"]

def typy_i_braki(X):
    X = X.replace([np.inf, -np.inf], np.nan).copy()
    # Kody PKD i form prawnych są kategoriami, mimo liczbowego zapisu.
    for col in CAT_CODES:
        if col in X:
            X[col] = X[col].map(lambda value: str(value) if pd.notna(value) else np.nan)
    return X

In [ ]:
df_temp.head()

# Outlier policy
Obcinamy cechy liczbowe do kwantyli 1% i 99% wyznaczonych wyłącznie podczas `fit`. Kategorie i flagi braków nie są obcinane. Klasa poniżej przenosi dotychczasową operację `quantile` → `clip` do pipeline’u.

In [ ]:
class QuantileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        self.num_cols_ = X.select_dtypes(include=np.number).columns
        self.lower_caps_ = X[self.num_cols_].quantile(self.lower)
        self.upper_caps_ = X[self.num_cols_].quantile(self.upper)
        return self

    def transform(self, X):
        X = X.copy()
        X[self.num_cols_] = X[self.num_cols_].clip(
            lower=self.lower_caps_, upper=self.upper_caps_, axis=1
        )
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

# Adaptacja Pipeline / ColumnTransformer z Lab 3 (komórka 19)
# oraz Lab 1 (komórka 15). Selekcja typów następuje po usunięciu kolumn.
def preprocessing_ohe():
    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
            ("sc", StandardScaler()),
        ]), make_column_selector(dtype_include=np.number)),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("ohe", ohe(drop="first", handle_unknown="ignore")),
        ]), make_column_selector(dtype_exclude=np.number)),
    ])
    return Pipeline([
        ("typy", FunctionTransformer(typy_i_braki, feature_names_out="one-to-one")),
        ("kolumny", UsunStaleIdentyczne()),
        ("cap", QuantileCapper()),
        ("pre", pre),
    ])

preprocessing = preprocessing_ohe()
Z_train = preprocessing.fit_transform(X_train, y_train)
Z_val = preprocessing.transform(X_val)
feature_names = preprocessing.get_feature_names_out()
print("Train po przygotowaniu:", Z_train.shape)
print("Validation po przygotowaniu:", Z_val.shape)
assert Z_train.shape[1] == Z_val.shape[1] == len(feature_names)
assert np.isfinite(Z_train).all() and np.isfinite(Z_val).all()
# Test zostanie przekształcony dopiero przy końcowej ocenie.
# CV musi dostać surowe X_train i cały pipeline (preprocessing + model), nie Z_train.

# Binning i WoE

Binning/WoE pozostawiamy do etapu wyboru modelu interpretowalnego. Gotowa implementacja `WoEBinner` znajduje się w `labs/Lab_1_scoring.ipynb`, komórka 31. Powinna zostać umieszczona w pipeline modelu i dopasowana na foldach treningowych, podobnie jak preprocessing powyżej. Nie wybieramy teraz cech na podstawie validation ani test.

In [ ]:
# Zapisujemy SUROWE podziały, aby CV mogło ponownie dopasować preprocessing.
# Pliki pickle są lokalnymi artefaktami; wczytujemy tylko własne, zaufane pliki.
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
for name, values in {
    "X_train": X_train, "X_val": X_val, "X_test": X_test,
    "y_train": y_train, "y_val": y_val, "y_test": y_test,
}.items():
    values.to_pickle(ARTIFACT_DIR / f"{name}.pkl")

split_labels = pd.Series(index=df.index, dtype="object", name="split")
for name, values in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    split_labels.loc[values.index] = name
assert split_labels.notna().all()
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
split_labels.rename_axis("row_index").to_csv(ARTIFACT_DIR / "splits.csv")
print("Zapisano surowe podziały oraz indeksy w:", ARTIFACT_DIR)